# 실습 3. OpenAI API 활용 프로그램 (Day 2 / 120분)

> 시나리오: **리뷰 감성 분류를 LLM 으로 다시 풀고, sklearn(실습 2)과 비교**
>
> 이 노트북은 외부 예제 없이 `openai` 패키지만으로 진행합니다.

## Task
1. 단발 호출 / 토큰·비용 / 스트리밍 (`.env` 의 `OPENAI_API_KEY`)
2. 리뷰 100개 긍/부정 분류 — `temperature=0`, JSON 응답 강제 → **실습 2 와 비교**
3. 비용 측정 (`response.usage`) → **1만 건 가정 시 비용 추산**

## 0) 준비 — `.env` 로드 + 클라이언트

In [16]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                 # .env 의 OPENAI_API_KEY 를 환경변수로 로드
client = OpenAI()             # 키는 환경변수에서 자동으로 읽음
MODEL = "gpt-6-luna"   # 호출마다 reasoning_effort="none" 을 함께 넣는다 (교재 5.3)
print("client ready:", MODEL)

client ready: gpt-6-luna


In [ ]:
import os
from dotenv import load_dotenv

print("환경변수 로드:", load_dotenv())
print("API Key 존재:", bool(os.getenv("OPENAI_API_KEY")))
print("사용 모델:", MODEL)

## 1) Task 1 — 단발 호출 + 토큰/비용 확인

In [22]:
resp = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    messages=[
        {"role": "system", "content": "당신은 간결하게 답하는 도우미입니다."},
        {"role": "user", "content": "고양이를 한 문장으로 묘사해."},
    ],
    temperature=0.1,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

고양이는 부드러운 발걸음과 호기심 많은 눈빛으로 주변을 탐험하는 사랑스러운 동물입니다.
usage: CompletionUsage(completion_tokens=32, prompt_tokens=35, total_tokens=67, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None))


In [ ]:
resp = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
messages = [
    {"role": "system", "content": "너는 친절한 상담사야."},
    {"role": "user", "content": "안녕"},
    {"role": "assistant", "content": "안녕하세요!"},
    {"role": "user", "content": "내가 방금 뭐라고 했지?"}
],
    temperature=0.7,
)
print(resp.choices[0].message.content)
print("usage:", resp.usage)    # prompt_tokens / completion_tokens / total_tokens

### 스트리밍 — 토큰이 생성되는 대로 출력

In [9]:
stream = client.chat.completions.create(
    model=MODEL,
    reasoning_effort="none",
    messages=[{"role": "user", "content": "LLM 을 13줄로 설명해줘."}],
    stream=True,
)
print(stream)
for chunk in stream:
    delta = chunk.choices[0].delta.content or ""
    print(delta, end="")
print()

1. LLM은 ‘대규모 언어 모델(Large Language Model)’의 약자입니다.
2. 많은 양의 텍스트를 학습해 언어의 패턴을 익힙니다.
3. 질문에 답하고 글을 쓰거나 요약할 수 있습니다.
4. 번역, 분류, 아이디어 제안 같은 작업도 수행합니다.
5. 보통 문장을 토큰이라는 작은 단위로 나누어 처리합니다.
6. 입력된 문맥을 바탕으로 다음에 올 토큰을 예측합니다.
7. 이런 예측을 반복해 문장이나 답변을 만들어 냅니다.
8. 학습 과정에서 문법과 표현, 여러 주제의 정보를 습득합니다.
9. 하지만 사람처럼 의식이나 감정을 가진 것은 아닙니다.
10. 학습 데이터에 오류나 편향이 있으면 답변에도 반영될 수 있습니다.
11. 그럴듯하게 말하면서도 사실과 다른 내용을 만들기도 합니다.
12. 따라서 중요한 정보는 신뢰할 만한 자료로 확인해야 합니다.
13. LLM은 편리한 도구이며, 사람의 판단을 돕는 데 활용됩니다.


## 2) Task 2 — 리뷰 100개 긍/부정 분류 (LLM)

실습 1의 산출물 `reviews_clean.parquet` 에서 100건을 샘플링해 분류한다.
- `temperature=0` (분류는 결정적이어야 함)
- **JSON 응답 강제** (`response_format`) — 파싱 안전

In [10]:
import pandas as pd

df = pd.read_parquet("../data/reviews_clean.parquet")
df = df.dropna(subset=["rating"]).copy()
df = df[df["rating"] != 3]                      # 중립 제외 (실습 2 와 동일 기준)
df["label"] = (df["rating"] >= 4).astype(int)   # 정답: 4~5 긍정(1), 1~2 부정(0)
sample = df.sample(100, random_state=42).reset_index(drop=True)
print(sample["label"].value_counts())
sample[["clean_text", "label"]].head()

label
0    51
1    49
Name: count, dtype: int64


,clean_text,label
0,가성비 최고입니다 또 살게요,1
1,생각보다 너무 작고 부실해요,0
2,배송이 정말 빨라서 깜짝 놀랐어요,1
3,사진과 색이 완전히 달라서 실망했어요 ㅠㅠ,0
4,배송이 일주일이나 걸렸습니다 별로,0


In [ ]:
import json

SYSTEM = (
    "너는 한국어 쇼핑몰 리뷰 감성 분류기다. "
    "입력 리뷰가 긍정이면 1, 부정이면 0 을 고른다. "
    'JSON 으로만 답한다: {"label": 0 또는 1}'
)

def classify(text: str) -> tuple[int, int]:   # (라벨, 사용 토큰) 을 함께 반환
    print(str)
    resp = client.chat.completions.create(
        model=MODEL,
        reasoning_effort="none",
        temperature=0,
        response_format={"type": "json_object"},   # JSON 강제
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": text},
        ],
    )
    data = json.loads(resp.choices[0].message.content)
    print(data)
    print("입력 토큰:", resp.usage.prompt_tokens)
    print("출력 토큰:", resp.usage.completion_tokens)
    return int(data["label"]), resp.usage.total_tokens   # 라벨 + 비용계산용 토큰

# 1건 점검 — (라벨, 토큰) 튜플이 찍힌다
label, used = classify(sample.loc[0, "clean_text"])
print("라벨:", label, "/ 토큰:", used)

<class 'str'>
{'label': 1}
입력 토큰: 71
출력 토큰: 8
라벨: 1 / 토큰: 79


In [14]:
preds, tokens = [], 0
for t in sample["clean_text"]:
    label, used = classify(t)
    preds.append(label)
    tokens += used
sample["pred"] = preds
print("총 토큰:", tokens)

<class 'str'>
입력 토큰: 71
출력 토큰: 11
<class 'str'>
입력 토큰: 72
출력 토큰: 11
<class 'str'>
입력 토큰: 75
출력 토큰: 11
<class 'str'>
입력 토큰: 78
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 75
출력 토큰: 11
<class 'str'>
입력 토큰: 76
출력 토큰: 11
<class 'str'>
입력 토큰: 75
출력 토큰: 11
<class 'str'>
입력 토큰: 75
출력 토큰: 11
<class 'str'>
입력 토큰: 72
출력 토큰: 11
<class 'str'>
입력 토큰: 80
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 71
출력 토큰: 11
<class 'str'>
입력 토큰: 78
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 77
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 80
출력 토큰: 11
<class 'str'>
입력 토큰: 73
출력 토큰: 11
<class 'str'>
입력 토큰: 76
출력 토큰: 11
<class 'str'>
입력 토큰: 74
출력 토큰: 11
<class 'str'>
입력 토큰: 77
출력 토큰: 11
<class 'str'>
입력 토큰: 78
출력 토큰: 11
<class 'str'>
입력 토큰: 77
출력 토큰: 11
<class 'str'>
입력 토큰: 75
출력 토큰: 11
<class 'str'>
입력 토큰: 74
출력 토큰: 11
<class 'str'>
입력 토큰: 71
출력 토큰: 11
<class 'str'>


### sklearn(실습 2) 과 비교 — 정확도

In [15]:
from sklearn.metrics import accuracy_score, classification_report

acc = accuracy_score(sample["label"], sample["pred"])
print(f"LLM 정확도: {acc:.3f}")
print(classification_report(sample["label"], sample["pred"], digits=3))
# TODO: 실습 2 sklearn 정확도와 한 표로 비교 (정확도 / 비용 / 속도 / 일관성)

LLM 정확도: 0.970
              precision    recall  f1-score   support

           0      1.000     0.941     0.970        51
           1      0.942     1.000     0.970        49

    accuracy                          0.970       100
   macro avg      0.971     0.971     0.970       100
weighted avg      0.972     0.970     0.970       100



## 3) Task 3 — 비용 측정 + 1만 건 추산

In [26]:
# GPT-6 Luna 단가 (100만 토큰 기준)
PRICE_IN = 0.10 / 1_000_000  # 입력 토큰당 달러
EXCHANGE_RATE = 1400          # 환율: 1달러 = 1,400원

# 비용 계산
avg_tokens = tokens / len(sample)
cost_100 = tokens * PRICE_IN * EXCHANGE_RATE

print(f"평균 토큰/건: {avg_tokens:.1f}")
print(f"100건 비용: {cost_100:.2f}원")
print(f"1,000건 추산: {cost_100 * 10:,.2f}원")
print(f"1만 건 추산: {cost_100 * 100:,.2f}원")

# TODO: ML vs LLM — 언제 무엇을 쓸지 결론 작성

평균 토큰/건: 85.8
100건 비용: 1.20원
1,000건 추산: 12.01원
1만 건 추산: 120.13원


## 회고 / 산출물
- [ ] 비교표: ML vs LLM (정확도 / 비용 / 속도 / 일관성)
- [ ] 한 줄 결론 — *대량·단순 분류는 ___, 유연·복잡 추론은 ___*